# Chẩn đoán tràn số fp16 của Qwen2.5-VL-3B

Lượt B0 dev (17/11/2026) có 6/409 câu sinh ra chuỗi `"!!!!…"` dài tới hết `max_new_tokens`; lượt sweep C3 có 1/252 câu
như vậy. Dấu hiệu: logits thành inf/NaN, giải mã tham lam chọn token id 0 (`"!"`). Vì crop đổi ảnh đầu vào, tập câu bị
tràn số khác nhau giữa các nhánh, nên lỗi này làm nhiễu so sánh B0/C3/C4/C5. Notebook này:

1. **Tái hiện**: chạy lại 6 câu lỗi và 6 câu đối chứng với đúng cấu hình B0 (fp16, eager, `max_pixels = 1024·28·28`, prompt p2).
2. **Định vị**: gắn forward hook lên mọi module, chạy prefill và 3 bước giải mã đầu, tìm module **đầu tiên** cho output
   không hữu hạn (vision encoder hay layer LLM nào) và độ lớn kích hoạt từng layer so với ngưỡng fp16 (65504).
3. **Thử cách sửa đồng loạt** (áp cho mọi câu, mọi nhánh, không chạy lại riêng câu lỗi): (a) vision tower fp32,
   (b) các layer LLM cuối + norm cuối fp32 (luồng residual giữ fp32 từ layer đó), (c) cả hai. Ghi số câu lỗi được sửa,
   số câu đối chứng đổi đáp án, VRAM đỉnh, thời gian/câu.

Chỉ dùng câu **dev**. Kết quả (17/11/2026): tràn số ở `visual.blocks.31.mlp.down_proj`; cách (a) được chọn và đã đưa vào
`src/vlm/qwen.py` (mặc định `vision_fp32=True`). Notebook này nạp model với `vision_fp32=False` để tái hiện lỗi gốc.

**Cần:** Kaggle dataset `da2-eval` (đã upload cho B0, không cần đóng gói lại), GPU T4, Internet On. Khoảng 15 phút.

**Kết quả:** tải `/kaggle/working/fp16_diag_results.zip`, giải nén vào gốc repo (`results/diag_fp16/`).

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils pillow numpy
import os, sys, shutil, glob

hits = glob.glob('/kaggle/input/**/src/vlm/qwen.py', recursive=True)
assert hits, 'Không thấy src/vlm/qwen.py: kiểm tra đã Add Input dataset da2-eval chưa'
src_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
print('Dataset root:', src_root)
shutil.copytree(src_root, '/kaggle/working/da2', dirs_exist_ok=True)
os.chdir('/kaggle/working/da2'); sys.path.insert(0, '.')

import torch, transformers
print(torch.cuda.device_count(), 'GPU:', torch.cuda.get_device_name(0))
print('torch', torch.__version__, '| transformers', transformers.__version__)

In [ ]:
import json, time, re, gc, math
from pathlib import Path
from PIL import Image
from src.vlm.qwen import load_qwen, answer, DEFAULT_MODEL, DEFAULT_MAX_PIXELS
from src.vlm.prompts import answer_suffix, MAX_NEW_TOKENS, PROMPT_VERSION
from src.data_prep import eval_sets

torch.manual_seed(0)

# 6 câu ra "!!!!…" trong lượt B0 dev (results/B0_base/dev_*.jsonl)
FAIL = ['textvqa_38125', 'textvqa_39480', 'vitextvqa_41190', 'vitextvqa_20852', 'vitextvqa_28956', 'vitextvqa_38079']
# Đối chứng: 3 câu đầu không lỗi của mỗi bộ trong cùng file B0
CONTROL = ['textvqa_36949', 'textvqa_37858', 'textvqa_35387', 'vitextvqa_20402', 'vitextvqa_342', 'vitextvqa_1709']
items = {it['qid']: it for ds in ('textvqa', 'vitextvqa') for it in eval_sets.load(ds, 'dev')}
CASES = [items[q] for q in FAIL + CONTROL]

out_dir = Path('results/diag_fp16'); out_dir.mkdir(parents=True, exist_ok=True)
CFG = dict(model=DEFAULT_MODEL, dtype='float16', attn='eager', max_pixels=DEFAULT_MAX_PIXELS,
           prompt_version=PROMPT_VERSION, max_new_tokens=MAX_NEW_TOKENS, decoding='greedy', seed=0,
           fail=FAIL, control=CONTROL, torch=torch.__version__, transformers=transformers.__version__,
           gpu=torch.cuda.get_device_name(0))
json.dump(CFG, open(out_dir / 'config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)


def is_degenerate(pred: str) -> bool:
    # Chuỗi chỉ gồm một ký tự lặp lại (thường là "!"), dài >= 16
    s = pred.replace(' ', '').replace('\n', '')
    return len(s) >= 16 and len(set(s)) == 1


def run_cases(model, tag):
    res = {}
    for it in CASES:
        img = Image.open(it['image']).convert('RGB')
        torch.cuda.reset_peak_memory_stats(); t0 = time.time()
        pred = answer(model, processor, img, it['question'], answer_suffix(it['dataset']), MAX_NEW_TOKENS)
        r = dict(pred=pred, degenerate=is_degenerate(pred), sec=round(time.time() - t0, 2),
                 vram_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
        res[it['qid']] = r
        print(f"{tag:16s} {it['qid']:16s} {'LỖI' if r['degenerate'] else 'ok '} {r['sec']:5.1f}s | {pred[:70]!r}")
    return res


model, processor = load_qwen(DEFAULT_MODEL, DEFAULT_MAX_PIXELS, 'eager', vision_fp32=False)

## 1. Tái hiện với cấu hình B0

In [ ]:
base = run_cases(model, 'fp16 (B0)')
n_fail = sum(base[q]['degenerate'] for q in FAIL)
print(f'\nTái hiện: {n_fail}/{len(FAIL)} câu lỗi; đối chứng lỗi: {sum(base[q]["degenerate"] for q in CONTROL)}/{len(CONTROL)}')
if n_fail < len(FAIL):
    print('Có câu không tái hiện: lỗi không tất định (phụ thuộc kernel/phiên bản). Ghi lại phiên bản ở config.json.')

## 2. Định vị module tràn số

Hook chạy khi module chạy xong, nên module con được ghi trước module cha; module đầu tiên (theo thứ tự ghi) có output
không hữu hạn là nơi tràn số bắt đầu. `step` 1 là prefill, 2–4 là ba token đầu được sinh.

In [ ]:
FP16_MAX = 65504.0


def tensors_of(out):
    if torch.is_tensor(out):
        return [out]
    vals = list(out.values()) if isinstance(out, dict) else list(out) if isinstance(out, (tuple, list)) else []
    return [v for v in vals if torch.is_tensor(v)]


class Tracer:
    # Ghi (tên module, bước, |giá trị| lớn nhất, có inf/NaN, dtype) cho output của mọi module con
    def __init__(self, model):
        self.records, self.step = [], 0
        self.handles = [model.register_forward_pre_hook(self._next_step)]
        for name, m in model.named_modules():
            if name and not isinstance(m, torch.nn.ModuleList):
                self.handles.append(m.register_forward_hook(self._hook(name)))

    def _next_step(self, m, args):
        self.step += 1

    def _hook(self, name):
        def f(m, args, out):
            ts = [t for t in tensors_of(out) if t.is_floating_point() and t.numel()]
            if not ts:
                return
            v = torch.stack([torch.stack([t.detach().abs().amax().float(), (~torch.isfinite(t)).any().float()])
                             for t in ts]).tolist()
            bad = any(b for _, b in v)
            amax = math.inf if bad else max(a for a, _ in v)
            self.records.append(dict(name=name, step=self.step, amax=amax, bad=bad, dtype=str(ts[0].dtype)))
        return f

    def remove(self):
        for h in self.handles:
            h.remove()


def build_inputs(it):
    # Giống hệt src/vlm/qwen.answer
    img = Image.open(it['image']).convert('RGB')
    content = [{'type': 'image'}, {'type': 'text', 'text': it['question'] + answer_suffix(it['dataset'])}]
    text = processor.apply_chat_template([{'role': 'user', 'content': content}], tokenize=False,
                                         add_generation_prompt=True)
    return processor(text=[text], images=[img], return_tensors='pt').to(model.device)


trace = {}
for it in CASES:
    inp = build_inputs(it)
    tr = Tracer(model)
    with torch.no_grad():
        gen = model.generate(**inp, max_new_tokens=4, do_sample=False, return_dict_in_generate=True, output_logits=True)
    tr.remove()
    logits0 = gen.logits[0][0]
    first_bad = next((r for r in tr.records if r['bad']), None)
    trace[it['qid']] = dict(records=tr.records, first_bad=first_bad, n_tokens=int(inp['input_ids'].shape[1]),
                            logits0_finite=bool(torch.isfinite(logits0).all()),
                            first_tokens=processor.batch_decode(gen.sequences[:, inp['input_ids'].shape[1]:])[0])
    fb = f"bước {first_bad['step']}: {first_bad['name']} ({first_bad['dtype']})" if first_bad else 'không có inf/NaN trong 4 bước đầu'
    print(f"{it['qid']:16s} {trace[it['qid']]['n_tokens']:5d} token | logits bước 1 hữu hạn: "
          f"{trace[it['qid']]['logits0_finite']!s:5s} | 4 token đầu {trace[it['qid']]['first_tokens']!r} | {fb}")

In [ ]:
def profile(records, pattern, step=1):
    # |giá trị| lớn nhất ở output của chính block/layer (không phải module con) tại bước cho trước
    prof = {}
    for r in records:
        m = re.search(pattern, r['name'])
        if r['step'] == step and m:
            prof[int(m.group(1))] = r['amax']
    return prof


def show_profile(title, pattern):
    print(f'\n{title}: |giá trị| lớn nhất ở output, prefill (ngưỡng fp16 = {FP16_MAX:.0f})')
    fail = [profile(trace[q]['records'], pattern) for q in FAIL]
    ctrl = [profile(trace[q]['records'], pattern) for q in CONTROL]
    for i in sorted(set().union(*fail, *ctrl)):
        f = max(p.get(i, 0) for p in fail); c = max(p.get(i, 0) for p in ctrl)
        flag = '  <-- inf/NaN' if math.isinf(f) else '  <-- > 50% ngưỡng' if f > FP16_MAX / 2 else ''
        print(f'  {i:3d}  lỗi max {f:10.1f}   đối chứng max {c:10.1f}{flag}')


show_profile('Vision blocks', r'(?:^|\.)blocks\.(\d+)$')
show_profile('LLM decoder layers', r'(?:^|\.)layers\.(\d+)$')

print('\nTop 10 module con có |giá trị| hữu hạn lớn nhất ở prefill (gộp mọi câu):')
for grp, qs in (('lỗi', FAIL), ('đối chứng', CONTROL)):
    best = {}
    for q in qs:
        for r in trace[q]['records']:
            if r['step'] == 1 and not r['bad']:
                best[r['name']] = max(best.get(r['name'], 0), r['amax'])
    print(f'  [{grp}]')
    for name, a in sorted(best.items(), key=lambda x: -x[1])[:10]:
        print(f'    {a:10.1f}  {name}')

json.dump(trace,
          open(out_dir / 'trace.json', 'w', encoding='utf-8'), ensure_ascii=False, default=str)

## 3. Thử cách sửa đồng loạt

Mỗi cách sửa nạp lại model rồi đổi một phần sang fp32; hook ép kiểu ở biên (vào fp32, ra fp16). Với (b), luồng residual
giữ fp32 từ layer bắt đầu tới hết norm cuối, chỉ ép về fp16 sau norm (giá trị đã nhỏ), vì nếu ép về fp16 sau mỗi layer
thì residual vẫn tràn. Layer bắt đầu lấy từ kết quả mục 2 (sớm hơn 2 layer cho có biên); chỉ dùng câu dev.

In [ ]:
def cast_floats(x, dtype):
    if torch.is_tensor(x):
        return x.to(dtype) if x.is_floating_point() else x
    if isinstance(x, dict):  # gồm cả ModelOutput
        for k in list(x.keys()):
            x[k] = cast_floats(x[k], dtype)
        return x
    if isinstance(x, (tuple, list)):
        return type(x)(cast_floats(v, dtype) for v in x)
    return x


def find_parts(model):
    named = dict(model.named_modules())
    visual = next(m for n, m in named.items() if n.split('.')[-1] == 'visual')
    lname = next(n for n, m in named.items() if n.split('.')[-1] == 'layers' and isinstance(m, torch.nn.ModuleList))
    norm = named[lname.rsplit('.', 1)[0] + '.norm'] if '.' in lname else named['norm']
    return visual, named[lname], norm


def upcast_span(first, mods, last):
    for m in mods:
        m.float()
    first.register_forward_pre_hook(lambda m, a, k: (cast_floats(a, torch.float32), cast_floats(k, torch.float32)),
                                    with_kwargs=True)
    last.register_forward_hook(lambda m, a, o: cast_floats(o, torch.float16))


def apply_fix(model, vision=False, tail=None):
    visual, layers, norm = find_parts(model)
    if vision:
        upcast_span(visual, [visual], visual)
    if tail is not None:
        upcast_span(layers[tail], list(layers[tail:]) + [norm], norm)
    return model


n_layers = len(find_parts(model)[1])
llm_bad = [int(m.group(1)) for q in FAIL if trace[q]['first_bad']
           for m in [re.search(r'(?:^|\.)layers\.(\d+)', trace[q]['first_bad']['name'])] if m]
vis_bad = [q for q in FAIL if trace[q]['first_bad'] and re.search(r'(?:^|\.)visual\.', trace[q]['first_bad']['name'])]
TAIL = max(min(llm_bad) - 2, 0) if llm_bad else n_layers - 8
print(f'Tràn số bắt đầu ở vision: {len(vis_bad)} câu; ở LLM: {len(llm_bad)} câu (layer {sorted(llm_bad)}). '
      f'Thử tail từ layer {TAIL}/{n_layers}.')

CANDIDATES = [dict(name='vision_fp32', vision=True, tail=None),
              dict(name=f'llm_tail{TAIL}_fp32', vision=False, tail=TAIL),
              dict(name=f'vision+llm_tail{TAIL}_fp32', vision=True, tail=TAIL)]

In [ ]:
fixes = {'fp16 (B0)': base}
for c in CANDIDATES:
    del model; gc.collect(); torch.cuda.empty_cache()
    model, processor = load_qwen(DEFAULT_MODEL, DEFAULT_MAX_PIXELS, 'eager', vision_fp32=False)
    apply_fix(model, c['vision'], c['tail'])
    print(f"\n=== {c['name']} ===")
    fixes[c['name']] = run_cases(model, c['name'][:16])

print('\nTóm tắt:')
print(f"{'cách':28s} {'lỗi còn lại':>11s} {'đối chứng đổi đáp án':>21s} {'VRAM đỉnh':>10s} {'s/câu (đối chứng)':>18s}")
for name, res in fixes.items():
    left = sum(res[q]['degenerate'] for q in FAIL)
    changed = sum(res[q]['pred'] != base[q]['pred'] for q in CONTROL)
    vram = max(r['vram_gb'] for r in res.values())
    sec = sum(res[q]['sec'] for q in CONTROL) / len(CONTROL)
    print(f'{name:28s} {left:>5d}/{len(FAIL):<5d} {changed:>10d}/{len(CONTROL):<10d} {vram:>8.2f}GB {sec:>14.2f}')

json.dump(dict(candidates=CANDIDATES, tail=TAIL, results=fixes),
          open(out_dir / 'fixes.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
shutil.make_archive('/kaggle/working/fp16_diag_results', 'zip', '.', 'results/diag_fp16')
print('\nTải /kaggle/working/fp16_diag_results.zip')